In [ ]:
import random
from datetime import datetime, timedelta

from faker import Faker
from pydeequ.analyzers import *
from pydeequ.analyzers import AnalysisRunner, Size, Completeness
from pydeequ.checks import Check, CheckLevel
from pydeequ.repository import FileSystemMetricsRepository, ResultKey
from pydeequ.verification import VerificationSuite
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, LongType, StringType, IntegerType, DateType, TimestampType

In [ ]:
def generate_users(fake: Faker, count: int):
    rows = []
    for i in range(1, count + 1):
        rows.append({
            "user_id": i,
            "email": fake.unique.email(),
            "name": fake.name(),
            "age": fake.random_int(min=10, max=100),
            "gender": random.choice(["M", "F"]),
            "job": fake.job(),
            "address": fake.address(),
            "signup": (datetime.now() - timedelta(days=random.randint(0, 365))).date(),
            "created_at": datetime.now()
        })
    return rows


user_schema = StructType([
    StructField("user_id", LongType(), False),
    StructField("email", StringType(), False),
    StructField("name", StringType(), False),
    StructField("age", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("job", StringType(), True),
    StructField("address", StringType(), True),
    StructField("signup", DateType(), True),
    StructField("created_at", TimestampType(), True)
])

In [ ]:
spark = SparkSession.builder \
    .appName("Example Deequ") \
    .master("spark://spark-master.mmix.io:7077") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio.mmix.io:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "mmix") \
    .config("spark.hadoop.fs.s3a.secret.key", "mmixmmix") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.sql.shuffle.partitions", "1") \
    .getOrCreate()

### Deequ Configuration

In [ ]:
repository = FileSystemMetricsRepository(spark, "s3a://mmix-prod-dataengineer-validation/deequ/sample/metrics/orders/metrics.json")
resultKey = ResultKey(spark, ResultKey.current_milli_time(), {"pipeline": "orders", "dataset": "orders", "env": "prod"})

In [ ]:
users = spark.createDataFrame(data=generate_users(Faker("ko_KR"), 1), schema=user_schema)

In [ ]:
check = Check(spark, CheckLevel.Error, "Basic data checks") \
    .hasSize(lambda x: x == 100) \
    .isComplete("id") \
    .isComplete("name") \
    .isComplete("age") \
    .isComplete("gender") \
    .isComplete("email")

check_result = VerificationSuite(spark) \
    .onData(users) \
    .addCheck(check) \
    .useRepository(repository) \
    .saveOrAppendResult(resultKey) \
    .run()

In [ ]:
analysis_result = AnalysisRunner(spark) \
    .onData(users) \
    .addAnalyzer(Size()) \
    .addAnalyzer(Completeness("id")) \
    .addAnalyzer(Completeness("name")) \
    .addAnalyzer(Completeness("age")) \
    .addAnalyzer(Correlation("height", "weight")) \
    .addAnalyzer(Completeness("gender")) \
    .addAnalyzer(Completeness("address")) \
    .addAnalyzer(Completeness("job")) \
    .addAnalyzer(Completeness("email")) \
    .useRepository(repository) \
    .saveOrAppendResult(resultKey) \
    .run()